<a href="https://colab.research.google.com/github/rodrigoviannasn/mat422-insulin-diabetes-project/blob/main/notebooks/02_modeling_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Comparison and Results

**Project:** Assessing the Value of Two-Hour Serum Insulin for Diabetes Classification

## Notebook Goal

This notebook will compare diabetes-classification models with and without two-hour serum insulin. It will test whether insulin improves prediction after the other available variables are considered.

## Pre-Specified Modeling Plan

The primary comparison will use the same predictors in two versions: one version without insulin and one version with insulin. The models will be logistic regression, elastic-net logistic regression, and random forest. Logistic regression will serve as the main easy-to-interpret baseline.

I will compare two data-preparation approaches. The first will retain the raw values as released. The second will treat zero values in glucose, blood pressure, skin thickness, insulin, and BMI as possible missing values. Missing-value processing will occur only within the training data for each fold, so the test data will not affect model training or tuning.

The main analysis will use five-fold nested stratified cross-validation. The outer folds will estimate model performance on data not used for training. The inner folds will select model settings. Models with and without insulin will use the same outer data splits for a fair comparison.

ROC-AUC will be the primary evaluation metric. Secondary metrics will include precision-recall AUC, balanced accuracy, sensitivity, specificity, Brier score, and calibration plots. Sensitivity and specificity will use a probability threshold of 0.50. I will also run a sensitivity analysis without glucose because the recorded outcome is related to glucose testing.

The analysis will focus on whether insulin provides a consistent improvement across models and preprocessing approaches. Results will describe prediction within this dataset only and will not be interpreted as causal or as medical advice.

## 1. Setup and Data

This section loads the unchanged dataset from GitHub and defines the fixed settings used throughout the model comparison.

In [1]:
# Import libraries, load the unchanged dataset, and define fixed settings.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

DATA_URL = (
    "https://raw.githubusercontent.com/rodrigoviannasn/"
    "mat422-insulin-diabetes-project/main/data/diabetes.csv"
)

RANDOM_STATE = 422
OUTER_SPLITS = 5
INNER_SPLITS = 3
CLASSIFICATION_THRESHOLD = 0.50

TARGET = "Outcome"

ALL_PREDICTORS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]

PREDICTORS_WITHOUT_INSULIN = [
    column for column in ALL_PREDICTORS
    if column != "Insulin"
]

POSSIBLE_MISSING_ZERO_COLUMNS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

FIGURES_DIR = Path("results/figures")
TABLES_DIR = Path("results/tables")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_URL)

assert df.shape == (768, 9), f"Unexpected dataset shape: {df.shape}"
assert set(df[TARGET].unique()) == {0, 1}, "Outcome must contain only 0 and 1."

X = df[ALL_PREDICTORS].copy()
y = df[TARGET].copy()

print("Dataset loaded successfully.")
print(f"Observations: {len(df)}")
print(f"Predictors with insulin: {len(ALL_PREDICTORS)}")
print(f"Predictors without insulin: {len(PREDICTORS_WITHOUT_INSULIN)}")
print(f"Outcome counts: {y.value_counts().sort_index().to_dict()}")
print(f"pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

Dataset loaded successfully.
Observations: 768
Predictors with insulin: 8
Predictors without insulin: 7
Outcome counts: {0: 500, 1: 268}
pandas version: 2.2.3
scikit-learn version: 1.6.1


## 2. Preprocessing Approaches

The analysis will compare raw released values with two sensitivity approaches that treat selected zero values as possible missing values. All preprocessing will be fitted inside cross-validation training folds later in this notebook to avoid information leakage.

In [2]:
# Define preprocessing pipelines without fitting them to the full dataset.

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


class ZeroToNaNTransformer(BaseEstimator, TransformerMixin):
    """Replace selected zero values with NaN without changing other columns."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_transformed = X.copy()

        for column in self.columns:
            if column in X_transformed.columns:
                X_transformed[column] = X_transformed[column].mask(
                    X_transformed[column] == 0,
                    np.nan,
                )

        return X_transformed


def make_preprocessor(feature_names, approach):
    """Create one preprocessing pipeline for a selected feature set."""

    zero_columns = [
        column for column in POSSIBLE_MISSING_ZERO_COLUMNS
        if column in feature_names
    ]

    if approach == "raw_zero":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

    if approach == "possible_missing_median":
        return Pipeline([
            ("zero_to_nan", ZeroToNaNTransformer(zero_columns)),
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ])

    if approach == "possible_missing_knn":
        return Pipeline([
            ("zero_to_nan", ZeroToNaNTransformer(zero_columns)),
            ("scaler_before_knn", StandardScaler()),
            (
                "imputer",
                KNNImputer(
                    n_neighbors=5,
                    weights="distance",
                    add_indicator=True,
                ),
            ),
        ])

    raise ValueError(f"Unknown preprocessing approach: {approach}")


preprocessing_plan = pd.DataFrame([
    {
        "Approach": "Raw zero values",
        "Description": "Keep values exactly as released.",
        "Purpose": "Primary baseline preprocessing approach.",
    },
    {
        "Approach": "Possible missing values with median imputation",
        "Description": (
            "Convert selected zero values to possible missing values, "
            "then use median imputation and missingness indicators."
        ),
        "Purpose": "Primary sensitivity analysis.",
    },
    {
        "Approach": "Possible missing values with KNN imputation",
        "Description": (
            "Convert selected zero values to possible missing values, "
            "then use distance-weighted KNN imputation."
        ),
        "Purpose": "Additional sensitivity analysis.",
    },
])

display(preprocessing_plan)

preprocessing_plan.to_csv(
    TABLES_DIR / "preprocessing_plan.csv",
    index=False,
)

,Approach,Description,Purpose
0,Raw zero values,Keep values exactly as released.,Primary baseline preprocessing approach.
1,Possible missing values with median imputation,Convert selected zero values to possible missi...,Primary sensitivity analysis.
2,Possible missing values with KNN imputation,Convert selected zero values to possible missi...,Additional sensitivity analysis.
